# Geocode multiple addresses in bulk


Bulk geocode addresses from .txt or .csv files, then display the geocoded results on a map with customizable pop-ups.


1. The first pattern focuses on batch geocoding multiple addresses from a .txt file in your workspace. The workflow implemented under this pattern is designed to geocode single line addresses available in a `.txt` file.
2. The second pattern focuses on adding a .csv file in your workspace as an item to the portal and then geocoding from the item. The workflow implemented under this pattern is designed to geocode single line addresses available in a column in a `.csv` file.

**Proceed with either the first OR second pattern below based on your input data (`.txt` file OR `.csv` file)**

> Note: If your input address data is in the form of list or dictionaries within the notebook as a cell, you can skip directly to use this [batch_geocode()](https://developers.arcgis.com/python/latest/api-reference/arcgis.geocoding.html#arcgis.geocoding.batch_geocode) method from the ArcGIS API for Python.


## 1. Import libraries and connect to GIS


In [ ]:
import pandas as pd
from arcgis.features import FeatureLayer
from arcgis.geocoding import batch_geocode, geocode_from_items
from arcgis.gis import GIS, ItemProperties, ItemTypeEnum
from arcgis.map.popups import FieldInfo

In [ ]:
gis = GIS("home")

## 2. Set inputs


**Please set the appropriate variable to `True` in the cell below based on the file type of your input data.**

E.g., if you are providing your input data as single line addresses in a `.txt` file, you want to set the `txt_file` variable to `True` and leave the `csv_file` variable as `False`.

**Correctly setting the input values here, streamlines execution of the rest of this workflow.**


In [ ]:
# Set to True if your input data is in a .txt file
txt_file = False
# Set to True if your input data is in a .csv file
csv_file = False

> Notes for `.csv` input data:
>
> 1. Titles for the CSV item and the geocoded Feature Layer item below will have to be unique (not identical to titles of items of similar types already in your organization) so as to avoid errors while adding a new item.
> 2. Step 2 (Geocode data) assumes that this data is not already available in your Organization and adds it as an item before geocoding the data. If the CSV item exists, either delete that item, or replace steps that define item properties and publish the data as item, with steps to `get()` the CSV item.


In [ ]:
if txt_file:
    txt_file_path = "<PATH-TO-TXT-FILE-IN-WORKSPACE-WITH-ADDRESSES-TO-GEOCODE>"
if csv_file:
    csv_file_path = "<PATH-TO-CSV-FILE-IN-WORKSPACE-WITH-ADDRESSES-TO-GEOCODE>"
    title = "<TITLE-FOR-CSV-ITEM-TO-CREATE>"
    output_title = "<TITLE-FOR-GEOCODED-FEATURE-LAYER-ITEM>"

## 3. Geocode data


In [ ]:
if txt_file:
    # Access each address in a list
    address_list = []
    with open(txt_file_path, "r") as f:
        address_list = [line.strip() for line in f]
    # print(address_list)   # uncomment this line if you wish to see the contents of this list

    # Geocode this list and save the result as a FeatureSet
    result_fs = batch_geocode(addresses=address_list, as_featureset=True)

    # Convert this to a Spatially Enabled DataFrame
    geocoded_df = result_fs.df
if csv_file:
    result_df = pd.read_csv(csv_file_path)
    # Define item properties
    root_folder = gis.content.folders.get()
    item_properties = ItemProperties(title=title, item_type=ItemTypeEnum.CSV)

    # Publish as item
    job = root_folder.add(
        item_properties=item_properties, file=csv_file_path, stream=True
    )
    item = job.result()

    # Geocode published CSV item
    geocoding_result = geocode_from_items(
        input_data=item, output_type="Feature Layer", output_name=output_title, gis=gis
    )

    # Extract geocoded data in a Spatially Enabled DataFrame
    geocoded_df = FeatureLayer(geocoding_result.url + "/0").query(as_df=True)

In [ ]:
geocoded_df.head()

## 4. Plot the results on a map and add a popup to the map


In [ ]:
# Create map
map1 = gis.map()
geocoded_df.spatial.plot(map1)
map1.zoom_to_layer(geocoded_df)

# Create popup
popup = map1.content.popup(0)
popup.disable_popup = False
fields = [FieldInfo(field_name=column_name) for column_name in geocoded_df.columns]
popup.edit("Address", field_infos=fields)

# Display map
map1